# Task 1: Data Preparation and QA Pair Generation

This notebook covers the following steps:
1. Downloading the textbook chapter 10.
2. Cleaning the extracted text.
3. Chunking the data.
4. Requesting your Gemini API key dynamically.
5. Generating 20 Question/Answer pairs using Gemini 3.0 Flash.

In [ ]:
!pip install pymupdf langchain langchain-text-splitters langchain-google-genai

In [ ]:
import os
import re
import json
import getpass
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

## 1. Load and clean the PDF text

In [ ]:
def clean_text(text):
    # Remove excessive newlines
    text = re.sub(r'\n+', '\n', text)
    # Remove standalone page numbers
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    # Remove simple headers/footers
    text = re.sub(r'CHAPTER \d+.*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'Speech and Language Processing.*$', '', text, flags=re.MULTILINE)
    return text.strip()

pdf_path = "10.pdf"
if not os.path.exists(pdf_path):
    !curl -o 10.pdf https://web.stanford.edu/~jurafsky/slp3/10.pdf
    
loader = PyMuPDFLoader(pdf_path)
documents = loader.load()

full_text = ""
for doc in documents:
    full_text += clean_text(doc.page_content) + "\n"

print(f"Total cleaned text length: {len(full_text)} characters.")

In [ ]:
# Chunking the text
text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=100)
chunks = text_splitter.create_documents([full_text])
print(f"Created {len(chunks)} chunks.")

print("\n--- Sample Chunk 1 ---")
print(chunks[0].page_content)

## 2. Dynamic API Key Input
Please enter your **Gemini API Key (using gemini-3.0-flash)** below. Do not hardcode the API key in the code block.

In [ ]:
# Interactive prompt for API key
api_key = getpass.getpass("Please enter your Gemini API Key: ")
os.environ["GEMINI_API_KEY"] = api_key

# Initialize Gemini model with the specified model version
llm = ChatGoogleGenerativeAI(model="gemini-3.0-flash", temperature=0)
print("LLM Initialized!")

## 3. Generate QA Pairs (20 Total)
We use Gemini to generate question-answer pairs strictly from our 20 selected text chunks.

In [ ]:
prompt = PromptTemplate.from_template(
    "Generate a clear, standalone Question and Answer pair based STRICTLY on the following text chunk.\n"
    "Format your output exactly as a JSON object with 'question' and 'ground_truth_answer' keys.\n\n"
    "Text Chunk:\n{chunk}\n\n"
    "Output JSON only:"
)

qa_chain = prompt | llm

qa_pairs = []
step = len(chunks) // 20
selected_indices = [i * step for i in range(20)]

for i, idx in enumerate(selected_indices):
    chunk_text = chunks[idx].page_content
    try:
        response = qa_chain.invoke({"chunk": chunk_text})
        
        response_text = response.content.strip()
        if response_text.startswith("```json"):
            response_text = response_text[7:]
        if response_text.endswith("```"):
            response_text = response_text[:-3]
            
        qa_pair = json.loads(response_text)
        qa_pair["naive_rag_answer"] = ""
        qa_pair["contextual_retrieval_answer"] = ""
        qa_pairs.append(qa_pair)
        print(f"Generated pair {i+1}/20.")
    except Exception as e:
        print(f"Error generating QA for chunk {idx}: {e}")

In [ ]:
output_path = "answer/response-st-126010-chapter-10.json"
os.makedirs("answer", exist_ok=True)
with open(output_path, "w") as f:
    json.dump(qa_pairs, f, indent=2)
    
print(f"Successfully generated {len(qa_pairs)} QA pairs to {output_path}")